# WildfireSpreadTS — Temporal Fusion U-Net (v3)

**Next-Day Wildfire Spread Prediction with Multi-Level Temporal Attention**

Based on: Gerard, Zhao & Sullivan (NeurIPS 2023 D&B)

### What's new in v3
1. **Per-channel normalization** (mean/std from training years)
2. **Copy data to local disk** (avoids Drive FUSE timeouts)
3. **Test time adjustment** (fair eval across window sizes)
4. **Extra test metrics** (Precision, Recall, IoU)
5. **CosineAnnealingLR scheduler** (same as Paolo's ViT+UNet)

### Experiments
- All levels: temporal attention at every encoder level + bottleneck
- Bottleneck only: attention only at bottleneck (like UTAE)

### Setup (matching the paper)
- Loss: Dice | Optimizer: AdamW (lr=0.001, wd=0.01)
- Scheduler: CosineAnnealingLR (eta_min=1e-6)
- Best model: lowest **validation loss**
- Metric: AP (primary) + F1, Precision, Recall, IoU
- **3-fold CV**: folds 0, 3, 10

## Mount Google Drive & Install Dependencies

In [ ]:
# !pip install segmentation-models-pytorch h5py

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Imports

In [ ]:
import os
import json
import glob
import shutil
import numpy as np
import h5py
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (average_precision_score, f1_score,
                             precision_score, recall_score, jaccard_score)

from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Configuration

In [ ]:
DATA_ROOT = "/content/drive/MyDrive/WildfireSpreadTS_10pct"
LOCAL_DATA_DIR = "/content/data_local"  # [NEW] local copy for speed

# 3-fold cross-validation
FOLDS = {
    0:  {'train': ['2018', '2019'], 'val': ['2020'], 'test': ['2021']},
    3:  {'train': ['2018', '2020'], 'val': ['2021'], 'test': ['2019']},
    10: {'train': ['2020', '2021'], 'val': ['2018'], 'test': ['2019']},
}

# Channel definitions
CHANNEL_NAMES = [
    "VIIRS_M11", "VIIRS_I2", "VIIRS_I1",          # 0-2: surface reflectance
    "NDVI", "EVI2",                                 # 3-4: vegetation
    "total_precipitation", "wind_speed",            # 5-6: weather
    "wind_direction", "min_temperature",            # 7-8: weather
    "max_temperature", "energy_release_comp",       # 9-10: weather/fire
    "specific_humidity",                            # 11: weather
    "slope", "aspect", "elevation",                 # 12-14: topography
    "PDSI",                                         # 15: drought
    "landcover",                                    # 16: land cover
    "forecast_precip", "forecast_wind_speed",       # 17-18: forecast
    "forecast_wind_dir", "forecast_temperature",    # 19-20: forecast
    "forecast_humidity",                            # 21: forecast
    "active_fire",                                  # 22: fire (also label)
]
N_CHANNELS = len(CHANNEL_NAMES)
FIRE_CHANNEL = 22

# [NEW] Max window size for test adjustment (paper Section 5.2)
# All test sets start from day 6 regardless of window_size,
# so models with different window sizes are evaluated on same targets
MAX_WINDOW_FOR_TEST = 5

## Copy data to local disk (avoids Drive FUSE timeouts)

In [ ]:
# [NEW] Reading HDF5 from Google Drive during training is slow and can
# cause random timeouts. Copying to Colab's local SSD fixes this.

def copy_data_to_local(data_root, local_dir):
    """Copy all HDF5 files from Drive to local disk."""
    if os.path.exists(local_dir) and len(os.listdir(local_dir)) > 0:
        print("Local data already exists, skipping copy.")
        return local_dir

    print("Copying data from Drive to local disk...")
    all_files = []
    for split in ['train', 'val', 'test']:
        split_dir = os.path.join(data_root, split)
        if not os.path.isdir(split_dir):
            continue
        for year in sorted(os.listdir(split_dir)):
            year_dir = os.path.join(split_dir, year)
            if not os.path.isdir(year_dir):
                continue
            for f in sorted(os.listdir(year_dir)):
                if f.endswith('.hdf5'):
                    src = os.path.join(year_dir, f)
                    # Store as local_dir/year/filename.hdf5 (flat by year)
                    dst_dir = os.path.join(local_dir, year)
                    dst = os.path.join(dst_dir, f)
                    all_files.append((src, dst, dst_dir))

    for src, dst, dst_dir in tqdm(all_files, desc="Copying files"):
        os.makedirs(dst_dir, exist_ok=True)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)

    # Verify
    for year in sorted(os.listdir(local_dir)):
        n = len([f for f in os.listdir(os.path.join(local_dir, year))
                 if f.endswith('.hdf5')])
        print(f"  {year}: {n} fires")

    return local_dir

copy_data_to_local(DATA_ROOT, LOCAL_DATA_DIR)

Copying data from Drive to local disk...


Copying files:   0%|          | 0/59 [00:00<?, ?it/s]

  2018: 17 fires
  2019: 7 fires
  2020: 20 fires
  2021: 15 fires


'/content/data_local'

## Discover all fires by year (from local copy)

In [ ]:
def discover_fires_by_year(data_dir):
    """
    Index fires by year from the local flat directory.
    Returns: {year_string: [list of hdf5 file paths]}
    """
    fires_by_year = defaultdict(list)
    for year in sorted(os.listdir(data_dir)):
        year_dir = os.path.join(data_dir, year)
        if not os.path.isdir(year_dir):
            continue
        for f in sorted(os.listdir(year_dir)):
            if f.endswith('.hdf5'):
                fires_by_year[year].append(os.path.join(year_dir, f))
    return dict(fires_by_year)

ALL_FIRES = discover_fires_by_year(LOCAL_DATA_DIR)
print("\nAll fires by year:")
for year in sorted(ALL_FIRES.keys()):
    print(f"  {year}: {len(ALL_FIRES[year])} fire events")
print(f"  Total: {sum(len(v) for v in ALL_FIRES.values())} fires")


All fires by year:
  2018: 17 fire events
  2019: 7 fire events
  2020: 20 fire events
  2021: 15 fire events
  Total: 59 fires


## Compute normalization stats from training years

In [ ]:
# [NEW] Per-channel normalization using mean/std from training data only.
# This is what the official repo does via get_means_stds_missing_values().
# We compute it ourselves to avoid depending on the repo.
#
# Important: stats are computed per FOLD because training years change.
# NaN values are excluded from the computation.

def compute_channel_stats(fires_by_year, train_years):
    """
    Compute per-channel mean and std from training years only.
    NaN values are excluded (not counted as zeros).

    Returns:
        means: (23,) array of per-channel means
        stds: (23,) array of per-channel stds
    """
    print(f"  Computing normalization stats from years {train_years}...")

    # Accumulate running stats to avoid loading everything into memory
    channel_sums = np.zeros(N_CHANNELS)
    channel_sq_sums = np.zeros(N_CHANNELS)
    channel_counts = np.zeros(N_CHANNELS)

    train_files = []
    for year in train_years:
        if year in fires_by_year:
            train_files.extend(fires_by_year[year])

    for fpath in tqdm(train_files, desc="  Stats", leave=False):
        with h5py.File(fpath, 'r') as f:
            data = f['data'][:]  # (T, 23, H, W)

        for c in range(N_CHANNELS):
            ch_data = data[:, c, :, :]
            valid_mask = ~np.isnan(ch_data)
            valid_vals = ch_data[valid_mask]
            channel_sums[c] += valid_vals.sum()
            channel_sq_sums[c] += (valid_vals ** 2).sum()
            channel_counts[c] += valid_mask.sum()

    means = channel_sums / np.maximum(channel_counts, 1)
    stds = np.sqrt(channel_sq_sums / np.maximum(channel_counts, 1) - means ** 2)

    # Avoid division by zero for constant channels
    stds[stds < 1e-6] = 1.0

    print(f"  Stats computed. Example — ch0 mean={means[0]:.4f}, std={stds[0]:.4f}")
    return means, stds


# Pre-compute stats for all folds (saves time later)
FOLD_STATS = {}
for fold_id, fold in FOLDS.items():
    print(f"\nFold {fold_id}:")
    means, stds = compute_channel_stats(ALL_FIRES, fold['train'])
    FOLD_STATS[fold_id] = {'means': means, 'stds': stds}


Fold 0:
  Computing normalization stats from years ['2018', '2019']...


  Stats:   0%|          | 0/24 [00:00<?, ?it/s]

  Stats computed. Example — ch0 mean=1926.6674, std=1128.2706

Fold 3:
  Computing normalization stats from years ['2018', '2020']...


  Stats:   0%|          | 0/37 [00:00<?, ?it/s]

  Stats computed. Example — ch0 mean=1944.4037, std=1124.6459

Fold 10:
  Computing normalization stats from years ['2020', '2021']...


  Stats:   0%|          | 0/35 [00:00<?, ?it/s]

  Stats computed. Example — ch0 mean=1912.8728, std=1178.7627


## Dataset class with normalization and test adjustment

In [ ]:
class WildfireSpreadDataset(Dataset):
    """
    WildfireSpreadTS dataset with per-channel normalization.

    Key features:
    - Loads by YEAR (supports any fold)
    - Per-channel normalization (mean/std from training years)
    - NaN replaced with channel mean (not 0)
    - Test time adjustment for fair eval across window sizes
    - Returns (window_size, 23, H, W) with temporal dim separate

    Args:
        fires_by_year: dict {year: [paths]}
        years: which years to include
        means, stds: per-channel normalization stats (from training years)
        window_size: input days
        crop_size: spatial crop
        augment: flips/rotations
        min_start_day: [NEW] skip early samples for fair test eval
    """

    def __init__(self, fires_by_year, years, means, stds,
                 window_size=5, crop_size=128, augment=False,
                 min_start_day=0):
        self.window_size = window_size
        self.crop_size = crop_size
        self.augment = augment
        self.min_start_day = min_start_day

        # Store normalization stats as (1, 23, 1, 1) for broadcasting
        self.means = means.reshape(1, N_CHANNELS, 1, 1).astype(np.float32)
        self.stds = stds.reshape(1, N_CHANNELS, 1, 1).astype(np.float32)

        # Collect files for specified years
        self.hdf5_files = []
        for year in years:
            if year in fires_by_year:
                self.hdf5_files.extend(fires_by_year[year])

        # Build samples
        self.samples = []
        self._build_samples()

    def _build_samples(self):
        """
        Create sliding window samples.

        min_start_day ensures fair evaluation across window sizes:
        - Paper Section 5.2: "testing starts on the target of day 6"
        - For window=5: first sample starts at t=0, target=day 5 → OK
        - For window=1: without adjustment, first target=day 1
          With min_start_day=4: first target=day 5 → same test targets
        """
        for fpath in self.hdf5_files:
            with h5py.File(fpath, 'r') as f:
                T = f['data'].shape[0]

            # Start from min_start_day to ensure fair eval
            start = max(0, self.min_start_day)
            for t in range(start, T - self.window_size):
                self.samples.append((fpath, t))

        print(f"    {len(self.samples)} samples from {len(self.hdf5_files)} fires"
              f" (min_start_day={self.min_start_day})")

    def _normalize(self, x):
        """
        Per-channel normalization: (x - mean) / std
        NaN values are first replaced with the channel mean (so they become 0
        after normalization), matching the official repo's behavior.
        """
        # Replace NaN with channel mean before normalizing
        for c in range(N_CHANNELS):
            nan_mask = np.isnan(x[:, c, :, :])
            x[:, c, :, :][nan_mask] = self.means[0, c, 0, 0]

        # Normalize
        x = (x - self.means) / self.stds
        return x

    def _crop(self, x, y, random=True):
        h, w = x.shape[-2], x.shape[-1]
        if h < self.crop_size or w < self.crop_size:
            pad_h = max(0, self.crop_size - h)
            pad_w = max(0, self.crop_size - w)
            x = np.pad(x, [(0,0),(0,0),(0,pad_h),(0,pad_w)], mode='constant')
            y = np.pad(y, [(0,0),(0,pad_h),(0,pad_w)], mode='constant')
            h, w = x.shape[-2], x.shape[-1]

        if random and self.augment:
            top = np.random.randint(0, h - self.crop_size + 1)
            left = np.random.randint(0, w - self.crop_size + 1)
        else:
            top = (h - self.crop_size) // 2
            left = (w - self.crop_size) // 2

        x = x[:, :, top:top+self.crop_size, left:left+self.crop_size]
        y = y[:, top:top+self.crop_size, left:left+self.crop_size]
        return x, y

    def _augment_data(self, x, y):
        if np.random.rand() > 0.5:
            x = np.flip(x, axis=-1).copy()
            y = np.flip(y, axis=-1).copy()
        if np.random.rand() > 0.5:
            x = np.flip(x, axis=-2).copy()
            y = np.flip(y, axis=-2).copy()
        k = np.random.randint(4)
        if k > 0:
            x = np.rot90(x, k, axes=(-2, -1)).copy()
            y = np.rot90(y, k, axes=(-2, -1)).copy()
        return x, y

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fpath, t_start = self.samples[idx]

        with h5py.File(fpath, 'r') as f:
            input_data = f['data'][t_start : t_start + self.window_size].astype(np.float32)
            target_fire = f['data'][t_start + self.window_size, FIRE_CHANNEL]

        # [NEW] Normalize input channels
        x = self._normalize(input_data)

        # Binarize target (NaN → 0, then threshold)
        target_fire = np.nan_to_num(target_fire, nan=0.0)
        y = (target_fire > 0).astype(np.float32)[np.newaxis, ...]

        x, y = self._crop(x, y, random=self.augment)
        if self.augment:
            x, y = self._augment_data(x, y)

        return torch.from_numpy(x.copy()), torch.from_numpy(y.copy())

## Test the dataset

In [ ]:
fold_config = FOLDS[0]
stats = FOLD_STATS[0]
print(f"Testing fold 0: train={fold_config['train']}, val={fold_config['val']}, test={fold_config['test']}")

print("Train:")
train_ds_test = WildfireSpreadDataset(
    ALL_FIRES, fold_config['train'], stats['means'], stats['stds'],
    window_size=5, crop_size=128, augment=True, min_start_day=0)
print("Val:")
val_ds_test = WildfireSpreadDataset(
    ALL_FIRES, fold_config['val'], stats['means'], stats['stds'],
    window_size=5, crop_size=128, augment=False, min_start_day=0)
print("Test (with adjustment):")
test_ds_test = WildfireSpreadDataset(
    ALL_FIRES, fold_config['test'], stats['means'], stats['stds'],
    window_size=5, crop_size=128, augment=False,
    min_start_day=MAX_WINDOW_FOR_TEST - 5)  # =0 for window=5

x, y = train_ds_test[0]
print(f"\nSample x shape: {x.shape}")
print(f"Sample y shape: {y.shape}")
print(f"x range: [{x.min():.2f}, {x.max():.2f}]  (should be roughly normalized)")
print(f"Fire pixels: {y.sum().item():.0f} / {y.numel()} ({100*y.mean().item():.3f}%)")

del train_ds_test, val_ds_test, test_ds_test

Testing fold 0: train=['2018', '2019'], val=['2020'], test=['2021']
Train:
    405 samples from 24 fires (min_start_day=0)
Val:
    281 samples from 20 fires (min_start_day=0)
Test (with adjustment):
    232 samples from 15 fires (min_start_day=0)

Sample x shape: torch.Size([5, 23, 128, 128])
Sample y shape: torch.Size([1, 128, 128])
x range: [-3.13, 39.52]  (should be roughly normalized)
Fire pixels: 0 / 16384 (0.000%)


## Temporal Attention Module

In [ ]:
class TemporalAttention(nn.Module):
    """
    Multi-head temporal attention: learns which days matter most.
    Input: (B, T, C, H, W) → Output: (B, C, H, W)
    """
    def __init__(self, channels, n_heads=4):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = channels // n_heads
        self.query = nn.Parameter(torch.randn(1, n_heads, self.head_dim, 1, 1))
        self.key_proj = nn.Conv2d(channels, channels, kernel_size=1, bias=False)
        self.value_proj = nn.Conv2d(channels, channels, kernel_size=1, bias=False)
        self.scale = self.head_dim ** -0.5

    def forward(self, x):
        B, T, C, H, W = x.shape
        x_flat = x.reshape(B * T, C, H, W)
        keys = self.key_proj(x_flat).reshape(B, T, self.n_heads, self.head_dim, H, W)
        values = self.value_proj(x_flat).reshape(B, T, self.n_heads, self.head_dim, H, W)
        scores = (keys * self.query.unsqueeze(0)).sum(dim=3) * self.scale
        attn_weights = F.softmax(scores, dim=1)
        fused = (values * attn_weights.unsqueeze(3)).sum(dim=1)
        return fused.reshape(B, C, H, W)

## Temporal Fusion U-Net Model

In [ ]:
class TemporalFusionUNet(nn.Module):
    """
    U-Net with temporal attention for multi-day wildfire prediction.

    attention_mode:
      'all_levels':     attention at every encoder level + bottleneck
      'bottleneck_only': attention only at bottleneck (like UTAE)
    """
    def __init__(self, in_channels=23, n_heads=4,
                 features=[64, 128, 256, 512],
                 attention_mode='all_levels'):
        super().__init__()
        self.attention_mode = attention_mode

        # Shared encoder
        self.encoder_blocks = nn.ModuleList()
        ch = in_channels
        for f in features:
            self.encoder_blocks.append(self._conv_block(ch, f))
            ch = f
        self.pool = nn.MaxPool2d(2, 2)

        # Bottleneck
        self.bottleneck = self._conv_block(features[-1], features[-1] * 2)

        # Temporal attention
        if attention_mode == 'all_levels':
            self.temporal_attns = nn.ModuleList([
                TemporalAttention(f, n_heads=n_heads) for f in features
            ])
        else:
            self.temporal_attns = None
        self.bottleneck_attn = TemporalAttention(features[-1] * 2, n_heads=n_heads)

        # Decoder
        self.up_convs = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()
        for f in reversed(features):
            self.up_convs.append(nn.ConvTranspose2d(f * 2, f, kernel_size=2, stride=2))
            self.dec_blocks.append(self._conv_block(f * 2, f))

        self.final = nn.Conv2d(features[0], 1, kernel_size=1)

    def _conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def _encode_single_day(self, x):
        features = []
        for block in self.encoder_blocks:
            x = block(x)
            features.append(x)
            x = self.pool(x)
        bottleneck = self.bottleneck(x)
        return features, bottleneck

    def forward(self, x):
        B, T, C, H, W = x.shape

        # Encode each day with shared encoder
        all_features = [[] for _ in range(len(self.encoder_blocks))]
        all_bottlenecks = []
        for t in range(T):
            feats, bn = self._encode_single_day(x[:, t])
            for lvl, feat in enumerate(feats):
                all_features[lvl].append(feat)
            all_bottlenecks.append(bn)

        for lvl in range(len(all_features)):
            all_features[lvl] = torch.stack(all_features[lvl], dim=1)
        all_bottlenecks = torch.stack(all_bottlenecks, dim=1)

        # Temporal fusion
        fused_bn = self.bottleneck_attn(all_bottlenecks)
        fused_features = []
        for lvl in range(len(all_features)):
            if self.attention_mode == 'all_levels':
                fused = self.temporal_attns[lvl](all_features[lvl])
            else:
                fused = all_features[lvl].mean(dim=1)
            fused_features.append(fused)

        # Decode
        x = fused_bn
        for i, (up, dec) in enumerate(zip(self.up_convs, self.dec_blocks)):
            x = up(x)
            skip = fused_features[-(i + 1)]
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1)
            x = dec(x)

        return self.final(x)

## Verify model shapes

In [ ]:
for mode in ['all_levels', 'bottleneck_only']:
    m = TemporalFusionUNet(in_channels=23, attention_mode=mode).to(device)
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    with torch.no_grad():
        dummy = torch.randn(2, 5, 23, 128, 128).to(device)
        out = m(dummy)
    print(f"{mode}: {n_params:,} params | {dummy.shape} → {out.shape}")
del m, dummy, out
torch.cuda.empty_cache()

all_levels: 33,844,609 params | torch.Size([2, 5, 23, 128, 128]) → torch.Size([2, 1, 128, 128])
bottleneck_only: 33,147,329 params | torch.Size([2, 5, 23, 128, 128]) → torch.Size([2, 1, 128, 128])


## Loss, metrics, training functions

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        p, t = probs.view(-1), targets.view(-1)
        intersection = (p * t).sum()
        return 1 - (2 * intersection + self.smooth) / (p.sum() + t.sum() + self.smooth)


def compute_metrics(all_probs, all_labels):
    """
    [NEW] Compute AP, F1, Precision, Recall, IoU — matching Paolo's test metrics.
    AP is the paper's primary metric.
    """
    probs = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    preds = (probs > 0.5).astype(int)

    ap = average_precision_score(labels, probs) if labels.sum() > 0 else 0.0
    f1 = f1_score(labels, preds, zero_division=0)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    iou = jaccard_score(labels, preds, zero_division=0)

    return {'AP': ap, 'F1': f1, 'Precision': precision, 'Recall': recall, 'IoU': iou}


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, n = 0, 0
    for x, y in tqdm(loader, desc="  Train", leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n += 1
    return total_loss / max(n, 1)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, n = 0, 0
    all_probs, all_labels = [], []
    for x, y in tqdm(loader, desc="  Eval", leave=False):
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += criterion(logits, y).item()
        n += 1
        all_probs.append(torch.sigmoid(logits).cpu().numpy().flatten())
        all_labels.append(y.cpu().numpy().flatten())
    metrics = compute_metrics(all_probs, all_labels)
    metrics['loss'] = total_loss / max(n, 1)
    return metrics

## Single fold training

In [ ]:
def train_single_fold(fold_id, attention_mode='all_levels', window_size=5,
                      num_epochs=20, batch_size=4, lr=1e-3, crop_size=128):
    fold = FOLDS[fold_id]
    stats = FOLD_STATS[fold_id]

    print(f"\n{'='*65}")
    print(f"FOLD {fold_id}: train={fold['train']}, val={fold['val']}, test={fold['test']}")
    print(f"  mode={attention_mode} | window={window_size} | lr={lr}")
    print(f"{'='*65}")

    # [NEW] Test time adjustment: ensure test targets start from same day
    # regardless of window_size (paper Section 5.2)
    test_min_start = MAX_WINDOW_FOR_TEST - window_size

    print("  Building datasets...")
    train_ds = WildfireSpreadDataset(
        ALL_FIRES, fold['train'], stats['means'], stats['stds'],
        window_size=window_size, crop_size=crop_size, augment=True,
        min_start_day=0)  # Training uses all data
    val_ds = WildfireSpreadDataset(
        ALL_FIRES, fold['val'], stats['means'], stats['stds'],
        window_size=window_size, crop_size=crop_size, augment=False,
        min_start_day=0)
    test_ds = WildfireSpreadDataset(
        ALL_FIRES, fold['test'], stats['means'], stats['stds'],
        window_size=window_size, crop_size=crop_size, augment=False,
        min_start_day=test_min_start)  # [NEW] Fair test eval

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=2, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             num_workers=2, pin_memory=True)

    # Model
    model = TemporalFusionUNet(
        in_channels=N_CHANNELS, n_heads=4,
        features=[64, 128, 256, 512],
        attention_mode=attention_mode,
    ).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Parameters: {n_params:,}")

    # Loss & optimizer
    criterion = DiceLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr,
                                   betas=(0.9, 0.999), weight_decay=0.01)

    # [NEW] Learning rate scheduler — same as Paolo's ViT+UNet
    # CosineAnnealing smoothly reduces lr from initial value to eta_min
    # over the course of training, which helps convergence
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=1e-6)

    # Training — save best on val loss (paper)
    best_val_loss = float('inf')
    save_path = f'/content/best_tfunet_fold{fold_id}_{attention_mode}.pth'
    history = {'train_loss': [], 'val_loss': [], 'val_ap': [], 'val_f1': [], 'lr': []}

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
        val_metrics = evaluate(model, val_loader, criterion)

        # [NEW] Step the scheduler after each epoch
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_metrics['loss'])
        history['val_ap'].append(val_metrics['AP'])
        history['val_f1'].append(val_metrics['F1'])
        history['lr'].append(current_lr)

        print(f"  Epoch {epoch:3d}/{num_epochs} | "
              f"TrLoss: {train_loss:.4f} | "
              f"VlLoss: {val_metrics['loss']:.4f} | "
              f"VlAP: {val_metrics['AP']:.4f} | "
              f"VlF1: {val_metrics['F1']:.4f} | "
              f"lr: {current_lr:.2e}")

        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']
            torch.save(model.state_dict(), save_path)
            print(f"    -> Best model (val_loss={best_val_loss:.4f})")

    # [NEW] Test with all metrics
    model.load_state_dict(torch.load(save_path))
    test_metrics = evaluate(model, test_loader, criterion)
    print(f"\n  TEST RESULTS:")
    print(f"    AP:        {test_metrics['AP']:.4f}")
    print(f"    F1:        {test_metrics['F1']:.4f}")
    print(f"    Precision: {test_metrics['Precision']:.4f}")
    print(f"    Recall:    {test_metrics['Recall']:.4f}")
    print(f"    IoU:       {test_metrics['IoU']:.4f}")

    return test_metrics, history

## 3-fold CV runner

In [ ]:
def run_3fold_experiment(attention_mode, window_size=5, num_epochs=20,
                         batch_size=4, lr=1e-3):
    print(f"\n{'#'*65}")
    print(f"# EXPERIMENT: {attention_mode} (window={window_size})")
    print(f"{'#'*65}")

    fold_results = {}
    fold_histories = {}

    for fold_id in FOLDS.keys():
        test_metrics, history = train_single_fold(
            fold_id=fold_id, attention_mode=attention_mode,
            window_size=window_size, num_epochs=num_epochs,
            batch_size=batch_size, lr=lr)
        fold_results[fold_id] = test_metrics
        fold_histories[fold_id] = history
        torch.cuda.empty_cache()

    # Summary stats
    metric_names = ['AP', 'F1', 'Precision', 'Recall', 'IoU']
    summary = {'per_fold': fold_results, 'histories': fold_histories}
    for m in metric_names:
        vals = [fold_results[f][m] for f in FOLDS.keys()]
        summary[f'mean_{m}'] = np.mean(vals)
        summary[f'std_{m}'] = np.std(vals)

    print(f"\n{'='*65}")
    print(f"SUMMARY — {attention_mode}")
    print(f"{'='*65}")
    for fold_id in FOLDS.keys():
        r = fold_results[fold_id]
        print(f"  Fold {fold_id:2d} (test={FOLDS[fold_id]['test']}): "
              f"AP={r['AP']:.4f}, F1={r['F1']:.4f}, IoU={r['IoU']:.4f}")
    print(f"  {'─'*55}")
    for m in metric_names:
        print(f"  {m:>10s}: {summary[f'mean_{m}']:.4f} ± {summary[f'std_{m}']:.4f}")

    return summary

## Experiment 1 — ALL levels

In [ ]:
bbresults_all = run_3fold_experiment(
    attention_mode='all_levels', window_size=5,
    num_epochs=20, batch_size=4, lr=1e-3)


#################################################################
# EXPERIMENT: all_levels (window=5)
#################################################################

FOLD 0: train=['2018', '2019'], val=['2020'], test=['2021']
  mode=all_levels | window=5 | lr=0.001
  Building datasets...
    405 samples from 24 fires (min_start_day=0)
    281 samples from 20 fires (min_start_day=0)
    232 samples from 15 fires (min_start_day=0)
  Parameters: 33,844,609


  Train:   0%|          | 0/101 [00:00<?, ?it/s]

  Eval:   0%|          | 0/71 [00:00<?, ?it/s]

  Epoch   1/20 | TrLoss: 0.9733 | VlLoss: 0.9665 | VlAP: 0.1097 | VlF1: 0.1694 | lr: 1.00e-03
    -> Best model (val_loss=0.9665)


  Train:   0%|          | 0/101 [00:00<?, ?it/s]

...

  Train:   0%|          | 0/145 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b44972dbd80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b44972dbd80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

  Eval:   0%|          | 0/58 [00:00<?, ?it/s]

  Epoch  12/20 | TrLoss: 0.7024 | VlLoss: 0.7969 | VlAP: 0.2465 | VlF1: 0.3604 | lr: 4.22e-04


  Train:   0%|          | 0/145 [00:00<?, ?it/s]

In [ ]:
# ── Results from fold 0 (already done) ──
fold_0_results = {'AP': 0.3154, 'F1': 0.4302, 'Precision': 0.4176, 'Recall': 0.4436, 'IoU': 0.2741}

# ── Fixed training function (num_workers=0, no warnings) ──
def train_single_fold_fixed(fold_id, attention_mode='all_levels', window_size=5,
                            num_epochs=20, batch_size=4, lr=1e-3, crop_size=128):
    fold = FOLDS[fold_id]
    stats = FOLD_STATS[fold_id]
    test_min_start = MAX_WINDOW_FOR_TEST - window_size

    print(f"\n{'='*65}")
    print(f"FOLD {fold_id}: train={fold['train']}, val={fold['val']}, test={fold['test']}")
    print(f"{'='*65}")

    print("  Building datasets...")
    train_ds = WildfireSpreadDataset(ALL_FIRES, fold['train'], stats['means'], stats['stds'],
        window_size=window_size, crop_size=crop_size, augment=True, min_start_day=0)
    val_ds = WildfireSpreadDataset(ALL_FIRES, fold['val'], stats['means'], stats['stds'],
        window_size=window_size, crop_size=crop_size, augment=False, min_start_day=0)
    test_ds = WildfireSpreadDataset(ALL_FIRES, fold['test'], stats['means'], stats['stds'],
        window_size=window_size, crop_size=crop_size, augment=False, min_start_day=test_min_start)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=0, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             num_workers=0, pin_memory=True)

    model = TemporalFusionUNet(in_channels=N_CHANNELS, n_heads=4,
        features=[64, 128, 256, 512], attention_mode=attention_mode).to(device)
    print(f"  Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    criterion = DiceLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.999), weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

    best_val_loss = float('inf')
    save_path = f'/content/best_tfunet_fold{fold_id}_{attention_mode}.pth'
    history = {'train_loss': [], 'val_loss': [], 'val_ap': [], 'val_f1': [], 'lr': []}

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
        val_metrics = evaluate(model, val_loader, criterion)
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_metrics['loss'])
        history['val_ap'].append(val_metrics['AP'])
        history['val_f1'].append(val_metrics['F1'])
        history['lr'].append(current_lr)

        print(f"  Epoch {epoch:3d}/{num_epochs} | "
              f"TrLoss: {train_loss:.4f} | VlLoss: {val_metrics['loss']:.4f} | "
              f"VlAP: {val_metrics['AP']:.4f} | VlF1: {val_metrics['F1']:.4f} | lr: {current_lr:.2e}")

        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']
            torch.save(model.state_dict(), save_path)
            print(f"    -> Best model (val_loss={best_val_loss:.4f})")

    model.load_state_dict(torch.load(save_path))
    test_metrics = evaluate(model, test_loader, criterion)
    print(f"\n  TEST: AP={test_metrics['AP']:.4f} | F1={test_metrics['F1']:.4f} | "
          f"Prec={test_metrics['Precision']:.4f} | Rec={test_metrics['Recall']:.4f} | IoU={test_metrics['IoU']:.4f}")
    return test_metrics, history

# ── Run only folds 3 and 10 ──
print("Running fold 3...")
fold_3_results, hist_3 = train_single_fold_fixed(fold_id=3, attention_mode='all_levels')
torch.cuda.empty_cache()

print("\nRunning fold 10...")
fold_10_results, hist_10 = train_single_fold_fixed(fold_id=10, attention_mode='all_levels')
torch.cuda.empty_cache()

# ── Print all results together ──
print("\n" + "="*65)
print("ALL LEVELS — 3-Fold Results")
print("="*65)
all_aps = [fold_0_results['AP'], fold_3_results['AP'], fold_10_results['AP']]
all_f1s = [fold_0_results['F1'], fold_3_results['F1'], fold_10_results['F1']]
print(f"  Fold 0  (test=2021): AP={fold_0_results['AP']:.4f}")
print(f"  Fold 3  (test=2019): AP={fold_3_results['AP']:.4f}")
print(f"  Fold 10 (test=2019): AP={fold_10_results['AP']:.4f}")
print(f"  Mean ± Std: AP = {np.mean(all_aps):.4f} ± {np.std(all_aps):.4f}")
print(f"              F1 = {np.mean(all_f1s):.4f} ± {np.std(all_f1s):.4f}")

Running fold 3...

FOLD 3: train=['2018', '2020'], val=['2021'], test=['2019']
  Building datasets...
    580 samples from 37 fires (min_start_day=0)
    232 samples from 15 fires (min_start_day=0)
    106 samples from 7 fires (min_start_day=0)
  Parameters: 33,844,609


  Train:   0%|          | 0/145 [00:00<?, ?it/s]

  Eval:   0%|          | 0/58 [00:00<?, ?it/s]

  Epoch   1/20 | TrLoss: 0.9727 | VlLoss: 0.9259 | VlAP: 0.2403 | VlF1: 0.2851 | lr: 1.00e-03
    -> Best model (val_loss=0.9259)


  Train:   0%|          | 0/145 [00:00<?, ?it/s]

...

  Train:   0%|          | 0/128 [00:00<?, ?it/s]

  Eval:   0%|          | 0/75 [00:00<?, ?it/s]

  Epoch  20/20 | TrLoss: 0.6325 | VlLoss: 0.6326 | VlAP: 0.1441 | VlF1: 0.2726 | lr: 7.15e-06


  Eval:   0%|          | 0/27 [00:00<?, ?it/s]


  TEST: AP=0.0854 | F1=0.2006 | Prec=0.1713 | Rec=0.2418 | IoU=0.1115

ALL LEVELS — 3-Fold Results
  Fold 0  (test=2021): AP=0.3154
  Fold 3  (test=2019): AP=0.0636
  Fold 10 (test=2019): AP=0.0854
  Mean ± Std: AP = 0.1548 ± 0.1139
              F1 = 0.2734 ± 0.1110


## Experiment 2 — BOTTLENECK only

In [ ]:
results_bn = run_3fold_experiment(
    attention_mode='bottleneck_only', window_size=5,
    num_epochs=20, batch_size=4, lr=1e-3)

## Persistence baseline (3-fold)

In [ ]:
def persistence_baseline_3fold(window_size=5):
    print("\nPersistence baseline (3-fold):")
    all_metrics = {m: [] for m in ['AP', 'F1', 'Precision', 'Recall', 'IoU']}
    fold_aps = []

    for fold_id, fold in FOLDS.items():
        stats = FOLD_STATS[fold_id]
        test_min_start = MAX_WINDOW_FOR_TEST - window_size

        ds = WildfireSpreadDataset(
            ALL_FIRES, fold['test'], stats['means'], stats['stds'],
            window_size=window_size, crop_size=128, augment=False,
            min_start_day=test_min_start)

        probs_list, labels_list = [], []
        for i in range(len(ds)):
            x, y = ds[i]
            # Use last day's fire channel as prediction
            # Note: fire channel is normalized, so threshold is different
            # We use the raw unnormalized prediction (fire > 0 in original space)
            # Since normalized, fire pixels will have value > some threshold
            # Simplest: just use the normalized fire channel values as scores
            fire_today = x[-1, FIRE_CHANNEL].numpy()
            probs_list.append(fire_today.flatten())
            labels_list.append(y.numpy().flatten())

        metrics = compute_metrics(probs_list, labels_list)
        print(f"  Fold {fold_id} (test={fold['test']}): AP={metrics['AP']:.4f}, F1={metrics['F1']:.4f}")
        for m in all_metrics:
            all_metrics[m].append(metrics[m])
        fold_aps.append(metrics['AP'])

    result = {'per_fold_AP': fold_aps}
    for m in all_metrics:
        result[f'mean_{m}'] = np.mean(all_metrics[m])
        result[f'std_{m}'] = np.std(all_metrics[m])
    print(f"  Mean AP: {result['mean_AP']:.4f} ± {result['std_AP']:.4f}")
    return result

persist_results = persistence_baseline_3fold()

## Comparison plot

In [16]:
def plot_final_comparison(results_all, results_bn, persist_results):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Mean AP bar chart
    models = ['Persistence', 'Bottleneck\nonly', 'All\nlevels']
    means = [persist_results['mean_AP'], results_bn['mean_AP'], results_all['mean_AP']]
    stds = [persist_results['std_AP'], results_bn['std_AP'], results_all['std_AP']]
    colors = ['#78909C', '#FF9800', '#2196F3']

    bars = axes[0].bar(models, means, yerr=stds, capsize=5, color=colors, alpha=0.85)
    for bar, m, s in zip(bars, means, stds):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.005,
                     f'{m:.3f}', ha='center', fontsize=10)
    axes[0].set_ylabel('Test Average Precision')
    axes[0].set_title('Mean AP ± Std (3-fold CV)')
    axes[0].grid(True, alpha=0.3, axis='y')

    # Per-fold comparison
    fold_ids = list(FOLDS.keys())
    x_pos = np.arange(len(fold_ids))
    width = 0.25
    axes[1].bar(x_pos - width, persist_results['per_fold_AP'], width,
                label='Persistence', color='#78909C')
    axes[1].bar(x_pos, [results_bn['per_fold'][f]['AP'] for f in fold_ids], width,
                label='Bottleneck only', color='#FF9800')
    axes[1].bar(x_pos + width, [results_all['per_fold'][f]['AP'] for f in fold_ids], width,
                label='All levels', color='#2196F3')
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels([f"Fold {f}\ntest={FOLDS[f]['test']}" for f in fold_ids], fontsize=9)
    axes[1].set_ylabel('Test AP')
    axes[1].set_title('AP per Fold')
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3, axis='y')

    # Training curves (fold 0)
    axes[2].plot(results_all['histories'][0]['val_ap'], label='All levels', color='#2196F3')
    axes[2].plot(results_bn['histories'][0]['val_ap'], label='Bottleneck only', color='#FF9800')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Val AP')
    axes[2].set_title('Validation AP — Fold 0')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    fig.suptitle('Temporal Fusion U-Net — 3-Fold Cross-Validation', fontsize=14)
    plt.tight_layout()
    plt.show()

plot_final_comparison(results_all, results_bn, persist_results)

NameError: name 'results_all' is not defined

## Full results table

In [ ]:
print("\n" + "="*75)
print("FINAL RESULTS — 3-Fold CV (Folds 0, 3, 10)")
print("="*75)
print(f"{'Model':<35} {'AP':>10} {'F1':>10} {'Prec':>10} {'Rec':>10} {'IoU':>10}")
print("-"*85)

for name, res in [('Persistence', persist_results),
                   ('TF-UNet (bottleneck only)', results_bn),
                   ('TF-UNet (all levels)', results_all)]:
    print(f"{name:<35} "
          f"{res['mean_AP']:.3f}±{res['std_AP']:.3f} "
          f"{res['mean_F1']:.3f}±{res['std_F1']:.3f} "
          f"{res.get('mean_Precision', 0):.3f}±{res.get('std_Precision', 0):.3f} "
          f"{res.get('mean_Recall', 0):.3f}±{res.get('std_Recall', 0):.3f} "
          f"{res.get('mean_IoU', 0):.3f}±{res.get('std_IoU', 0):.3f}")

print("="*85)
print("\nPaper reference (full dataset, 12-fold CV, window=5):")
print("  Persistence:     AP = 0.193 ± 0.065")
print("  ResNet18 U-Net:  AP = 0.344 ± 0.076")
print("  UTAE:            AP = 0.372 ± 0.088")
print("\nNote: Our results are on 10% subset with 3 folds.")